In [2]:
from datasets import load_dataset

ds = load_dataset("Yinpei/robomme_preprocessed_data")

/Users/gengyifan/miniforge3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['messages', 'objects', 'images', 'videos'],
        num_rows: 147952
    })
})


In [4]:
train_ds = ds['train']
print(len(train_ds))
print(train_ds.features)

147952
{'messages': List({'role': Value('string'), 'content': Value('string')}), 'objects': {'ref': List(Value('null')), 'bbox': List(List(Value('int64')))}, 'images': List(Value('string')), 'videos': List(Value('string'))}


In [5]:
from pprint import pprint

example = train_ds[0]

print(example.keys())
print()

for k, v in example.items():
    print(f"===== {k} =====")
    print(type(v))

    if isinstance(v, list):
        print(f"length: {len(v)}")
        if len(v) > 0:
            print("first element type:", type(v[0]))
            print("first element:")
            pprint(v[0])

    elif isinstance(v, dict):
        pprint(v)

    else:
        print(v)

    print()

dict_keys(['messages', 'objects', 'images', 'videos'])

===== messages =====
<class 'list'>
length: 3
first element type: <class 'dict'>
first element:
{'content': 'You are a robot program that predicts actions. The current input '
            'images from the front-view camera shows the most recent actions '
            'the robot has executed. The past keyframes are selected frames of '
            'particular importance from all the actions the robot has executed '
            'so far. Based on these, output the current subtask the robot '
            'should execute and nothing else. Some tasks may have a video '
            'input for initial setup, some may not.\n'
            '\n'
            'Return a JSON with:\n'
            '- current_subtask: the action that should be executed at the '
            'current timestep\n'
            '- keyframe_positions: list of frame positions (1-indexed) from '
            'the current input images where actions change',
 'role': 'system'}


In [6]:
pprint(example['messages'])

[{'content': 'You are a robot program that predicts actions. The current input '
             'images from the front-view camera shows the most recent actions '
             'the robot has executed. The past keyframes are selected frames '
             'of particular importance from all the actions the robot has '
             'executed so far. Based on these, output the current subtask the '
             'robot should execute and nothing else. Some tasks may have a '
             'video input for initial setup, some may not.\n'
             '\n'
             'Return a JSON with:\n'
             '- current_subtask: the action that should be executed at the '
             'current timestep\n'
             '- keyframe_positions: list of frame positions (1-indexed) from '
             'the current input images where actions change',
  'role': 'system'},
 {'content': 'The task has a video input for initial setup: <video>\n'
             'The task goal is: watch the video carefully, then us

In [7]:
for i in range(100):
    sample = train_ds[i]

    user_msg = sample["messages"][1]["content"]

    if "particular importance:[" in user_msg:
        if "[]" not in user_msg:
            print(i)
            print(user_msg[:1000])
            break

1
The task has a video input for initial setup: <video>
The task goal is: watch the video carefully, then use the stick attached to the robot to retrace the same pattern
Here are the selected frames from the entirety of the full execution that are of particular importance:[<image>]
Here is current input image list from the front-view camera: [<image>, <image>, <image>, <image>, <image>, <image>, <image>, <image>]

What subtask should the robot execute and what is the keyframe position?


In [2]:
from datasets import load_dataset

ds = load_dataset("Yinpei/robomme_data_h5")

In [1]:
# H5 Episode Loader
# episode > timesteps > (obs, action, done)

import h5py
import numpy as np
from pathlib import Path

def load_episode(h5_path):
    traj = []

    with h5py.File(h5_path, "r") as f:
        for ep in f.keys():
            ep_grp = f[ep]

            ts_keys = sorted(
                [k for k in ep_grp.keys() if k.startswith("timestep_")],
                key=lambda x: int(x.split("_")[-1])
            )

            for t in ts_keys:
                ts = ep_grp[t]

                obs = np.concatenate([
                    np.array(ts["obs"]["eef_state"]),
                    np.array(ts["obs"]["joint_state"]),
                    np.array(ts["obs"]["gripper_state"]),
                ])

                action = np.array(ts["action"]["joint_action"])

                traj.append({
                    "episode": ep,
                    "timestep": int(t.split("_")[-1]),
                    "obs": obs,
                    "action": action,
                })

    return traj


ModuleNotFoundError: No module named 'h5py'